# 07 — Robustness: repeated-CV gate + seed-bagged CatBoost
Public LB (0.83026) ≈ OOF (0.82898): the model is **not** overfit, but headroom is tiny and the error structure shows the weak Deep class is diffuse, confident overlap (partly irreducible synthetic noise). So the highest-value moves left are not a cleverer model but **(1)** a de-noised metric we can trust for accept/reject decisions, and **(2)** a lower-variance final model for the private 70% split.

- **Repeated CV gate** — `RepeatedStratifiedKFold(5×5)` → mean ± std OOF macro-F1. The std is the noise floor: only keep a future change if its OOF gain clearly beats it.
- **Seed-bagged CatBoost** — average `predict_proba` over K=9 seeds of the *exact* deployed config (textbook bagging from the course). Pure variance reduction; cannot widen the CV–LB gap.

In [1]:
# --- Shared toolbox (identical across notebooks; see build_notebooks.py) ---
import warnings, json
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
N_FOLDS = 5
CLASS_NAMES = {0: "Wake", 1: "Light", 2: "Deep", 3: "REM"}
CLASSES = np.array([0, 1, 2, 3])
EOG = "eog_burst_index"            # the only column with missing values (~50%)

RAW_FEATURES = [
    "eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
    "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power", "eeg_spectral_entropy",
    "eeg_spindle_density", "eeg_kcomplex_rate", "emg_chin_tone", "emg_tone_variance",
    "eog_movement_density", "eog_amplitude", "heart_rate_mean", "heart_rate_variability",
    "respiration_rate", "respiration_variability", "spo2_mean", "body_movement_index",
    EOG,
]
POWER = ["eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
         "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power"]

HERE = Path.cwd()
ART = HERE / "artifacts"; ART.mkdir(exist_ok=True)
SUB = HERE / "submissions"; SUB.mkdir(exist_ok=True)


def load_data():
    """Return (train_df, test_df). Features kept as-is (NaN preserved)."""
    tr = pd.read_csv(HERE / "train.csv")
    te = pd.read_csv(HERE / "test.csv")
    return tr, te


def macro_f1(y_true, y_pred):
    """Competition metric: macro-averaged F1 over the 4 classes."""
    return f1_score(y_true, y_pred, average="macro")


def per_class_f1(y_true, y_pred):
    f = f1_score(y_true, y_pred, average=None, labels=CLASSES)
    return {CLASS_NAMES[c]: round(float(f[i]), 4) for i, c in enumerate(CLASSES)}


def softplus(x):
    """Numerically stable log(1+exp(x)); strictly positive and monotonic.
    Used to turn z-scored band powers (~50% negative) into positive magnitudes
    so band ratios are well-defined instead of dividing by near-zero."""
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def _aligned_proba(model, X):
    """predict_proba with columns aligned to CLASSES = [0,1,2,3]."""
    p = model.predict_proba(X)
    cls = list(np.asarray(model.classes_))
    idx = [cls.index(c) for c in CLASSES]
    return p[:, idx]


def run_oof(make_model, X, y, X_test, folds, needs_impute=False, use_eval_set=False):
    """Out-of-fold training under fixed folds.

    Returns (oof, test_p, oof_macro, fold_scores):
      oof     : (n_train, 4) out-of-fold probabilities (each row predicted once)
      test_p  : (n_test, 4) test probabilities, averaged over the 5 fold-models
      oof_macro: global macro-F1 over the assembled OOF matrix (primary metric)

    Two model families, identical folds:
      - CatBoost (needs_impute=False): NaN passed through natively.
      - sklearn trees (needs_impute=True): add EOG-missing flag + fill EOG NaN
        with the TRAIN-FOLD median (fit on train fold only -> no leakage)."""
    n = len(y)
    oof = np.zeros((n, len(CLASSES)))
    test_p = np.zeros((len(X_test), len(CLASSES)))
    fold_scores = []
    for tr_idx, va_idx in folds:
        Xtr, Xva, Xte = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy(), X_test.copy()
        ytr, yva = y[tr_idx], y[va_idx]
        if needs_impute:
            med = Xtr[EOG].median()
            for d in (Xtr, Xva, Xte):
                if EOG + "_missing" not in d.columns:
                    d[EOG + "_missing"] = d[EOG].isna().astype("int8")
                d[EOG] = d[EOG].fillna(med)
            assert not Xtr.isna().any().any(), "NaN remained after impute"
        model = make_model()
        if use_eval_set:
            model.fit(Xtr, ytr, eval_set=(Xva, yva))
        else:
            model.fit(Xtr, ytr)
        oof[va_idx] = _aligned_proba(model, Xva)
        test_p += _aligned_proba(model, Xte) / len(folds)
        fold_scores.append(macro_f1(yva, oof[va_idx].argmax(1)))
    oof_macro = macro_f1(y, oof.argmax(1))
    return oof, test_p, oof_macro, fold_scores


def load_folds():
    """Load the fixed StratifiedKFold split saved by 02_baseline."""
    d = np.load(ART / "folds.npz", allow_pickle=True)
    return [(d[f"tr{i}"], d[f"va{i}"]) for i in range(N_FOLDS)]


In [2]:
def log_result(step, model, feature_set, oof_macro, pcf, notes=""):
    """Write one row to results_log.csv. Idempotent per (step, model): re-running
    a notebook replaces its own row rather than duplicating it."""
    import csv
    path = HERE / "results_log.csv"
    header = ["step", "model", "feature_set", "oof_macro_f1",
              "f1_Wake", "f1_Light", "f1_Deep", "f1_REM", "notes"]
    rows = []
    if path.exists():
        with open(path, newline="") as fh:
            data = list(csv.reader(fh))
        if data and data[0] == header:
            rows = [r for r in data[1:] if not (len(r) >= 2 and r[0] == step and r[1] == model)]
    row = [step, model, feature_set, round(float(oof_macro), 5),
           pcf["Wake"], pcf["Light"], pcf["Deep"], pcf["REM"], notes]
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(header); w.writerows(rows); w.writerow(row)
    print("logged:", step, model, "OOF macro-F1 =", round(float(oof_macro), 5))


In [3]:
from catboost import CatBoostClassifier
cols = json.load(open(ART / "feature_cols.json"))["columns"]
Xtr = pd.DataFrame(np.load(ART / "features_v1_train.npy"), columns=cols)
Xte = pd.DataFrame(np.load(ART / "features_v1_test.npy"), columns=cols)
y = np.load(ART / "y_train.npy")
test_ids = np.load(ART / "test_ids.npy")
folds = load_folds()
# Use the EXACT config sub01 deployed (tuning kept the defaults).
bp = json.load(open(ART / "catboost_best_params.json"))
print("deployed CatBoost config:", bp, "| features_v1:", Xtr.shape[1], "cols")

def make_cat(seed=SEED):
    kw = dict(loss_function="MultiClass", eval_metric="TotalF1:average=Macro",
        iterations=3000, learning_rate=bp["lr"], depth=bp["depth"], l2_leaf_reg=bp["l2"],
        random_seed=seed, od_type="Iter", od_wait=150, use_best_model=True,
        thread_count=-1, allow_writing_files=False, verbose=False)
    if bp.get("balanced"):
        kw["auto_class_weights"] = "Balanced"
    return CatBoostClassifier(**kw)

deployed CatBoost config: {'depth': 6, 'lr': 0.03, 'l2': 3.0, 'balanced': False} | features_v1: 28 cols


## 1. Repeated-CV gate (`RepeatedStratifiedKFold(5×5)`)
Single 5-fold OOF noise is on the same order as the gains we chase (thousandths), so selecting on one split risks picking a lucky-split model that regresses on the private LB. We run 5 independent 5-fold splits (25 fits) and report the mean and **std** of the global OOF macro-F1. That std is the accept/reject threshold for every later experiment.

In [4]:
from sklearn.model_selection import RepeatedStratifiedKFold
N_REPEATS = 5
rskf = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=SEED)
splits = list(rskf.split(np.zeros(len(y)), y))
repeat_macro, repeat_pcf = [], []
for r in range(N_REPEATS):
    oof = np.zeros((len(y), len(CLASSES)))
    for tr_idx, va_idx in splits[r*N_FOLDS:(r+1)*N_FOLDS]:
        m = make_cat()
        m.fit(Xtr.iloc[tr_idx], y[tr_idx], eval_set=(Xtr.iloc[va_idx], y[va_idx]))
        oof[va_idx] = _aligned_proba(m, Xtr.iloc[va_idx])
    s = macro_f1(y, oof.argmax(1))
    repeat_macro.append(s); repeat_pcf.append(per_class_f1(y, oof.argmax(1)))
    print(f"  repeat {r+1}: OOF macro-F1 = {s:.5f}")
gate_mean, gate_std = float(np.mean(repeat_macro)), float(np.std(repeat_macro))
pc_mean = {c: round(float(np.mean([d[c] for d in repeat_pcf])), 4)
           for c in ["Wake", "Light", "Deep", "REM"]}
print(f"\nGATE: OOF macro-F1 = {gate_mean:.5f} +/- {gate_std:.5f}  "
      f"(over {N_REPEATS} repeats x {N_FOLDS} folds)")
print(f"=> accept a future change only if its OOF gain clearly exceeds ~{gate_std:.4f} (1 sigma).")
print("mean per-class F1:", pc_mean)
json.dump({"repeat_macro": [round(x, 5) for x in repeat_macro], "mean": round(gate_mean, 5),
           "std": round(gate_std, 5), "n_repeats": N_REPEATS, "n_folds": N_FOLDS},
          open(ART / "repeated_cv_gate.json", "w"), indent=2)
log_result("07_robustness", "catboost_repeatedCV", "features_v1", gate_mean, pc_mean,
           f"GATE mean+/-std over {N_REPEATS}x{N_FOLDS}; std={gate_std:.5f}")

  repeat 1: OOF macro-F1 = 0.82898


  repeat 2: OOF macro-F1 = 0.82389


  repeat 3: OOF macro-F1 = 0.82509


  repeat 4: OOF macro-F1 = 0.82627


  repeat 5: OOF macro-F1 = 0.82567

GATE: OOF macro-F1 = 0.82598 +/- 0.00169  (over 5 repeats x 5 folds)
=> accept a future change only if its OOF gain clearly exceeds ~0.0017 (1 sigma).
mean per-class F1: {'Wake': 0.8513, 'Light': 0.8492, 'Deep': 0.7734, 'REM': 0.8301}
logged: 07_robustness catboost_repeatedCV OOF macro-F1 = 0.82598


## 2. Seed-bagged CatBoost
**Diagnostic first:** train K=9 seeds and look at the *single-model* per-seed OOF spread. If the spread is tiny the averaging ceiling is tiny too — we then bag for **stability**, not for score. The bag averages `predict_proba` across all seeds (OOF) and across folds×seeds (test). All seeds are fixed and logged, so the result is fully reproducible.

In [5]:
SEEDS = [42, 7, 13, 101, 202, 303, 404, 505, 2024]   # K=9, fixed & logged
K = len(SEEDS)
seed_oofs = {s: np.zeros((len(y), len(CLASSES))) for s in SEEDS}
sb_test = np.zeros((len(Xte), len(CLASSES)))
for tr_idx, va_idx in folds:
    Xt, Xv = Xtr.iloc[tr_idx], Xtr.iloc[va_idx]
    yt, yv = y[tr_idx], y[va_idx]
    for s in SEEDS:
        m = make_cat(seed=s)
        m.fit(Xt, yt, eval_set=(Xv, yv))
        seed_oofs[s][va_idx] = _aligned_proba(m, Xv)
        sb_test += _aligned_proba(m, Xte) / (len(folds) * K)

# diagnostic: how much does the seed alone move the OOF score?
per_seed = {s: macro_f1(y, seed_oofs[s].argmax(1)) for s in SEEDS}
vals = np.array(list(per_seed.values()))
print("per-seed single-model OOF macro-F1:")
for s in SEEDS:
    print(f"   seed {s:>4}: {per_seed[s]:.5f}")
print(f"spread -> min {vals.min():.5f}  max {vals.max():.5f}  std {vals.std():.5f}")

# the bag
sb_oof = sum(seed_oofs.values()) / K
sb_macro = macro_f1(y, sb_oof.argmax(1))
single = per_seed[SEED]
print(f"\nsingle-seed (42): {single:.5f}")
print(f"seed-bag (K={K}):  {sb_macro:.5f}   ({sb_macro - single:+.5f} vs single seed)")
print("seed-bag per-class F1:", per_class_f1(y, sb_oof.argmax(1)))
np.save(ART / "seedbag_oof.npy", sb_oof)
np.save(ART / "seedbag_test.npy", sb_test)
json.dump({"seeds": SEEDS, "per_seed_macro": {str(s): round(per_seed[s], 5) for s in SEEDS},
           "seed_std": round(float(vals.std()), 5), "single_seed42": round(single, 5),
           "bag_macro": round(float(sb_macro), 5)},
          open(ART / "seedbag.json", "w"), indent=2)
log_result("07_robustness", f"catboost_seedbag_K{K}", "features_v1", sb_macro,
           per_class_f1(y, sb_oof.argmax(1)),
           f"K={K} seeds; single={single:.5f}; seed_std={vals.std():.5f}")

per-seed single-model OOF macro-F1:
   seed   42: 0.82898
   seed    7: 0.82268
   seed   13: 0.82691
   seed  101: 0.82549
   seed  202: 0.82429
   seed  303: 0.82599
   seed  404: 0.82203
   seed  505: 0.82390
   seed 2024: 0.82437
spread -> min 0.82203  max 0.82898  std 0.00203

single-seed (42): 0.82898
seed-bag (K=9):  0.82415   (-0.00483 vs single seed)
seed-bag per-class F1: {'Wake': 0.849, 'Light': 0.8468, 'Deep': 0.7726, 'REM': 0.8282}
logged: 07_robustness catboost_seedbag_K9 OOF macro-F1 = 0.82415


## Seed-bag submission (`sub03`)
Shipped as a stability-hardened candidate **regardless** of OOF lift — variance reduction is private-LB insurance, not a score chase. The deterministic single-seed CatBoost (`sub01`) remains the documented reference.

In [6]:
name3 = f"sub03_CatBoostSeedBagK{K}_GBDT.csv"
pred3 = sb_test.argmax(1).astype(int)
sub3 = pd.DataFrame({"id": test_ids, "sleep_stage": pred3})
assert sub3.shape == (5000, 2)
assert sub3["id"].tolist() == list(range(9000, 14000))
assert sub3["sleep_stage"].isin(CLASSES).all()
sub3.to_csv(SUB / name3, index=False)
print("wrote", SUB / name3, "| OOF macro-F1:", round(sb_macro, 5))
print("class counts:", sub3["sleep_stage"].value_counts().sort_index().to_dict())
# divergence from the single-CatBoost primary (sub01)
single_test = np.load(ART / "catboost_tuned_test.npy")
diff = int((sb_test.argmax(1) != single_test.argmax(1)).sum())
print(f"seed-bag vs single-CatBoost test predictions differ on {diff} of 5000 rows")

wrote C:\Users\rasul\Documents\workspace_ws\yessenov_data_lab_program\day9\submissions\sub03_CatBoostSeedBagK9_GBDT.csv | OOF macro-F1: 0.82415
class counts: {0: 1116, 1: 1314, 2: 1261, 3: 1309}
seed-bag vs single-CatBoost test predictions differ on 77 of 5000 rows


### Takeaways
- The repeated-CV **gate** (mean ± std) is now the accept/reject rule for every future experiment — no more chasing sub-noise gains on a single split.
- The **seed-bag** trades a few minutes of compute for lower private-LB variance with no overfit risk; if the per-seed spread is < ~0.002 the OOF number barely moves, which is the expected (and honest) outcome near this ceiling.
- Submissions to keep as the private-LB pair: `sub01` (deterministic single CatBoost, the validated reference) and `sub03` (seed-bagged, variance-reduced).